Name: **Stephane Chambrelain Talla Fotsing**  
Student Number: **ST139766**  
Program: **Data Analytics and Artificial Intelligence**  
Course - **Introduction to Artificial Intelligence**  
**Assignment 13: Generative AI Essentials**

In [12]:
import requests
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
import textwrap

#### Model Architecture

GPTs use a decoder-only Transformer architecture. Attention mechanisms help them identify relevant relationships between tokens, while position information preserves their order. During training, the model predicts the next token using only preceding tokens and adjusts its weights to reduce prediction errors.

To generate text, GPT converts a prompt into tokens, which can represent words, word parts, or punctuation. It calculates probabilities for the next token, selects one, and adds it to the sequence. This process repeats until the response is complete or a length limit is reached.

#### load the book

In [15]:
tf.keras.utils.set_random_seed(42)

url = "https://www.gutenberg.org/ebooks/11.txt.utf-8"

response = requests.get(url, timeout=60)
response.raise_for_status()

text = response.content.decode("utf-8-sig")

# Remove the website's header and footer
start = text.index("*** START OF")
text = text[text.index("\n", start):]
text = text.split("*** END OF")[0]

# Lowercase and remove extra spaces
text = " ".join(text.lower().split())

print(textwrap.fill(text[:500], width=80))

[illustration] alice’s adventures in wonderland by lewis carroll the millennium
fulcrum edition 3.0 contents chapter i. down the rabbit-hole chapter ii. the
pool of tears chapter iii. a caucus-race and a long tale chapter iv. the rabbit
sends in a little bill chapter v. advice from a caterpillar chapter vi. pig and
pepper chapter vii. a mad tea-party chapter viii. the queen’s croquet-ground
chapter ix. the mock turtle’s story chapter x. the lobster quadrille chapter xi.
who stole the tarts? chap


#### Convert characters into numbers

In [7]:
characters = sorted(set(text))

char_to_id = {char: i for i, char in enumerate(characters)}
id_to_char = dict(enumerate(characters))

encoded = np.array([char_to_id[char] for char in text])

print("Unique characters:", len(characters))

Unique characters: 48


#### Create training examples

In [8]:
sequence_length = 40
X = []
y = []

for i in range(0, len(encoded) - sequence_length, 5):
    X.append(encoded[i:i + sequence_length])
    y.append(encoded[i + sequence_length])

X = np.array(X)
y = np.array(y)

print("Training examples:", X.shape)

Training examples: (28618, 40)


#### Build and train the model

In [9]:
model = Sequential([
    Input(shape=(sequence_length,)),
    Embedding(input_dim=len(characters), output_dim=32),
    LSTM(64),
    Dense(len(characters), activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

history = model.fit(X, y, epochs=10, batch_size=64)

Epoch 1/10
448/448 ━━━━━━━━━━━━━━━━━━━━ 16s 31ms/step - loss: 2.7953
Epoch 2/10
448/448 ━━━━━━━━━━━━━━━━━━━━ 14s 31ms/step - loss: 2.3913
Epoch 3/10
448/448 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - loss: 2.2825
Epoch 4/10
448/448 ━━━━━━━━━━━━━━━━━━━━ 14s 31ms/step - loss: 2.2043
Epoch 5/10
448/448 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - loss: 2.1427
Epoch 6/10
448/448 ━━━━━━━━━━━━━━━━━━━━ 14s 31ms/step - loss: 2.0916
Epoch 7/10
448/448 ━━━━━━━━━━━━━━━━━━━━ 14s 31ms/step - loss: 2.0482
Epoch 8/10
448/448 ━━━━━━━━━━━━━━━━━━━━ 14s 30ms/step - loss: 2.0103
Epoch 9/10
448/448 ━━━━━━━━━━━━━━━━━━━━ 21s 30ms/step - loss: 1.9761
Epoch 10/10
448/448 ━━━━━━━━━━━━━━━━━━━━ 20s 30ms/step - loss: 1.9452


#### Generate text from a seed

In [14]:
def generate_text(seed, length=200):
    result = seed.lower()

    if not result or any(char not in char_to_id for char in result):
        raise ValueError("Use a non-empty seed with characters from the book.")

    for _ in range(length):
        # Keep the last 40 characters; pad short seeds with spaces
        context = result[-sequence_length:].rjust(sequence_length)
        numbers = np.array([[char_to_id[char] for char in context]])

        probabilities = model(numbers, training=False).numpy()[0]
        probabilities = probabilities.astype("float64")
        probabilities /= probabilities.sum()

        next_id = np.random.choice(len(characters), p=probabilities)
        result += id_to_char[next_id]

    return result

generated_text = generate_text("alice was ")
print(textwrap.fill(generated_text, width=80))

alice was the kntun!” ash shank in weep? of oly qu,” “ald’t ar uid ol the kid
tule, aid af tadide aimisebe u be doptle is she houdinne! gor cow. i wand so a
toke, “mou doen leat ohete aij on that she tmeam ugse


#### Implementation

In [13]:
# Ask the user for a story starter
seed = input("Enter the beginning of your story: ")

# Generate a continuation using the trained model
story = generate_text(seed, length=300)

print("\n--- Generated Story Draft ---\n")
print(textwrap.fill(story, width=80))

Enter the beginning of your story: alice walked into the garden and saw

--- Generated Story Draft ---

alice walked into the garden and sawh his withed: a hi—bmiyeing ath”n; “whe what
i rithlink bat the is loolled,” “nis’r, caid an! you dadaice ding thatl: ‘uh
mire taifh; and he ham the mangeaf it i’s a fage was yeey.” anid. “eigh, the
moch ay the ond,” “ant aplt the timen the hen.” so but as out has oor thilg.
his, she whats oli uthes


In [1]:
!find "/content/drive/MyDrive" -iname "Assignment_13.ipynb"

find: ‘/content/drive/MyDrive’: No such file or directory


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!find "/content/drive/MyDrive" -iname "Assignment_13.ipynb"

/content/drive/MyDrive/Colab Notebooks/Assignment_13.ipynb


In [4]:
!git clone https://github.com/StephaneTF/colab-git-assignment2-ST.git /content/colab-git-assignment2-ST

Cloning into '/content/colab-git-assignment2-ST'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 77 (delta 29), reused 28 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 3.23 MiB | 18.76 MiB/s, done.
Resolving deltas: 100% (29/29), done.


In [5]:
!ls -la /content/colab-git-assignment2-ST

total 3788
drwxr-xr-x 7 root root    4096 Sep 19 09:26  .
drwxr-xr-x 1 root root    4096 Sep 19 09:26  ..
drwxr-xr-x 2 root root    4096 Sep 19 09:26  Assignment_14
-rw-r--r-- 1 root root  490404 Sep 19 09:26  Assignment_15.ipynb
-rw-r--r-- 1 root root    5415 Sep 19 09:26  assignment2.ipynb
drwxr-xr-x 2 root root    4096 Sep 19 09:26  Assignment_8
drwxr-xr-x 2 root root    4096 Sep 19 09:26  Assignment_9
drwxr-xr-x 8 root root    4096 Sep 19 09:26  .git
drwxr-xr-x 2 root root    4096 Sep 19 09:26  Lesson_10_assignment
-rw-r--r-- 1 root root   12704 Sep 19 09:26  Lesson_3_assignment.ipynb
-rw-r--r-- 1 root root   80271 Sep 19 09:26 'Lesson 4 assignment.ipynb'
-rw-r--r-- 1 root root  549515 Sep 19 09:26  Lesson_5_Assignment.ipynb
-rw-r--r-- 1 root root 2321005 Sep 19 09:26  Lesson_6_ml_basic_assignment.ipynb
-rw-r--r-- 1 root root  376413 Sep 19 09:26  Lesson_7_assignment.ipynb
